# DS 301 Final Project: Credit Card Default Prediction
**Team Members:** Chung Vong (Simon), Maria, Eduardo, Gabriel  

## Project Overview
This project implements a machine learning pipeline to predict credit card defaults. We reproduce the baseline methodology from the research paper *"Credit Default Mining Using Combined Machine Learning and Heuristic Approach"* by Islam et al., specifically focusing on the K-Nearest Neighbors (KNN) algorithm.

To fulfill the project contribution requirements, we upgrade the baseline by applying `SMOTE` to handle the highly imbalanced dataset, introducing `Logistic Regression` and `Decision Trees`, and utilizing `GridSearchCV` for hyperparameter optimization to maximize the recall score.

In [6]:
# Import necessary libraries for data processing and machine learning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

In [10]:
# 1. Load the dataset (Fixed path for Google Colab)
file_name = 'default-of-credit-card-clients.csv'

try:
    df = pd.read_csv(file_name, header=1)
    if 'ID' not in df.columns:
        df = pd.read_csv(file_name)
except:
    df = pd.read_csv(file_name)

# Standardize the target column name
if 'default.payment.next.month' in df.columns:
    df = df.rename(columns={'default.payment.next.month': 'default'})
elif 'default payment next month' in df.columns:
    df = df.rename(columns={'default payment next month': 'default'})

# 2. Separate features (X) and target variable (y)
X = df.drop(['ID', 'default'], axis=1)
y = df['default']

# 3. Feature Scaling using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print("Data Preprocessing Completed successfully.")
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Data Preprocessing Completed successfully.
Training data shape: (24000, 23)
Testing data shape: (6000, 23)


In [11]:
# Helper function to evaluate and print model metrics
def evaluate_model(model, model_name, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    print(f"--- {model_name} ---")
    print(f"Accuracy:  {accuracy_score(y_te, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_te, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_te, y_pred):.4f}\n")
    return y_pred

print("PART 1: REPRODUCING PAPER BASELINE\n")
# Reproducing only the KNN model as it was taught in class and used in the paper
y_pred_knn = evaluate_model(KNeighborsClassifier(), "K-Nearest Neighbors (KNN Baseline)", X_train, y_train, X_test, y_test)

PART 1: REPRODUCING PAPER BASELINE

--- K-Nearest Neighbors (KNN Baseline) ---
Accuracy:  0.7927
Recall:    0.3557
F1-Score:  0.4314



In [12]:
print("PART 2: OUR CONTRIBUTIONS & IMPROVEMENTS\n")

# Contribution 1: Handling Class Imbalance using SMOTE
print("Applying SMOTE to balance the training data...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Contribution 2: Implementing Logistic Regression (Not in paper)
evaluate_model(LogisticRegression(random_state=42, max_iter=1000), "Logistic Regression (with SMOTE)", X_train_smote, y_train_smote, X_test, y_test)

# Contribution 3: Hyperparameter Tuning using GridSearchCV on Decision Tree
print("--- Decision Tree Tuning via GridSearchCV ---")
param_grid = {
    'max_depth': [3, 5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='recall')
grid_search.fit(X_train_smote, y_train_smote)

print(f"Best Parameters Found: {grid_search.best_params_}")
y_pred_tuned = grid_search.predict(X_test)
print(f"Tuned Decision Tree Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"Tuned Decision Tree Recall:   {recall_score(y_test, y_pred_tuned):.4f}")

PART 2: OUR CONTRIBUTIONS & IMPROVEMENTS

Applying SMOTE to balance the training data...
--- Logistic Regression (with SMOTE) ---
Accuracy:  0.6703
Recall:    0.6255
F1-Score:  0.4563

--- Decision Tree Tuning via GridSearchCV ---
Best Parameters Found: {'max_depth': 15, 'min_samples_split': 2}
Tuned Decision Tree Accuracy: 0.7242
Tuned Decision Tree Recall:   0.5147


## 3. Project Conclusion & Findings

Based on the execution of the machine learning models, our team observed the following critical insights:

* **Baseline Reproduction:** The K-Nearest Neighbors (KNN) model from the original research paper achieved a relatively high overall accuracy (around 79.2%). However, it suffered from a very low recall score (around 35.5%). This indicates that while the model is accurate overall, it fails to identify the majority of actual credit card defaults due to the highly imbalanced nature of the dataset.
* **Our Contributions:** To address this limitation, we introduced **SMOTE** to balance the training data. We then applied and evaluated models taught in class but not prioritized in the paper, specifically **Logistic Regression** and **Decision Trees**, alongside **GridSearchCV** for optimal hyperparameter tuning.
* **Significant Improvements:** Our methodology successfully upgraded the baseline approach. The Logistic Regression model trained on SMOTE data drastically improved the recall score to **62.55%**. Similarly, the tuned Decision Tree achieved a recall of **51.47%**.
* **Business Impact:** In the context of credit risk mining, maximizing recall is far more important than raw accuracy, because the financial cost of missing a true default account heavily outweighs the cost of a false positive. Our contributed models successfully prioritize the identification of high-risk customers, aligning perfectly with practical financial risk management goals.